# 03 — Phase A: probe gate + fusion-head training

- **GATE 1** (go/no-go): linear probes on masked-mean-pooled features per modality.
  Helium-only macro-F1 ≥ 0.55 (chance = 0.25) means the hidden states carry the emotion.
- **Fusion head**: Helium→256 + Mimi latent→128 + prosody→32 → 2-layer causal GRU(256)
  → 4 logits (~2.0M params). Per-step CE on agent frames + turn-pooled CE, modality
  dropout, label smoothing, class weights, cosine LR, early stop on val macro-F1.
- **GATE 2**: fusion ≥ best probe + 0.05.
- Reports: confusion matrix, per-class P/R, user-emotion leakage probe, ECE +
  temperature scaling, smoothed switches/min.


In [ ]:
from pathlib import Path
import os, sys, json, time
import numpy as np
import torch

sys.path.insert(0, os.getcwd())
import eh_common as eh

P = eh.get_paths()   # override: P = eh.get_paths("/your/root") or set EMOTION_HEAD_ROOT
eh.mode_banner(P)    # FULL vs SMOKE_TEST — set once in NB00 (run_config.json)
RUN = eh.run_profile(P)   # all size knobs (full | smoke) — see eh_common.run_profile
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16
print("ROOT:", P.ROOT, "| device:", DEVICE)
for k in ("MOSHI_BNB4", "MIMI_FILE", "SPM_FILE", "VOICE_PT"):
    p = Path(getattr(P, k))
    print(f"  {k}: {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
import collections
BATCH_SIZE = RUN.nb03_batch_size   # full 16 | smoke 4
EPOCHS = RUN.nb03_epochs           # full 12 | smoke 2
LR = 3e-4
PATIENCE = RUN.nb03_patience       # full 3 | smoke 2
GATE1_MIN_F1 = 0.55      # helium-only probe macro-F1 (chance = 0.25)
GATE2_MIN_GAIN = 0.05    # fusion must beat the best probe by this
GATE_STRICT = False      # True -> assert gates; False -> print warnings only
if P.SMOKE:
    print("SMOKE: gates are informational only — tiny data is not representative")


In [ ]:
recs = list(eh.iter_shard_records(P.FEAT_A, "phaseA"))
print("clips:", len(recs), "| splits:", collections.Counter(r["split"] for r in recs))
by_split = {}
for r in recs:
    by_split.setdefault(r["split"], []).append(r)
tr = by_split.get("train", [])
va = by_split.get("val", [])
ho = by_split.get("holdout", [])
print(f"train {len(tr)} | val {len(va)} | holdout {len(ho)}")
assert tr and (va or ho), "not enough data — run 02 first"


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def pooled(rec, key):
    X = rec[key].float()
    m = rec["agent_mask"]
    return (X[m].mean(0) if m.any() else X.mean(0)).numpy()

def probe(train_recs, eval_recs, key):
    Xtr = np.stack([pooled(r, key) for r in train_recs])
    ytr = np.array([r["label"] for r in train_recs])
    Xev = np.stack([pooled(r, key) for r in eval_recs])
    yev = np.array([r["label"] for r in eval_recs])
    clf = LogisticRegression(max_iter=2000, n_jobs=-1).fit(Xtr, ytr)
    return float(f1_score(yev, clf.predict(Xev), average="macro"))

eval_recs = va or ho
f1_helium = probe(tr, eval_recs, "hidden")
f1_mimi = probe(tr, eval_recs, "mimi_lat")
f1_pros = probe(tr, eval_recs, "pros")
print(f"GATE 1 probes (macro-F1): helium={f1_helium:.3f} mimi={f1_mimi:.3f} "
      f"prosody={f1_pros:.3f} (chance=0.25)")
gate1 = f1_helium >= GATE1_MIN_F1
print("GATE 1:", "PASS" if gate1 else "FAIL — Helium hidden states may not carry emotion")
if GATE_STRICT:
    assert gate1, "stop here and reconsider the approach"


In [ ]:
class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        return (r["hidden"], r["mimi_lat"], r["pros"], r["agent_mask"], int(r["label"]))

def collate(batch):
    T = max(b[0].shape[0] for b in batch)
    B = len(batch)
    H = torch.zeros(B, T, eh.HELIUM_DIM)
    M = torch.zeros(B, T, eh.MIMI_LATENT_DIM)
    PR = torch.zeros(B, T, eh.PROSODY_DIM)
    MK = torch.zeros(B, T, dtype=torch.bool)
    LB = torch.zeros(B, dtype=torch.long)
    for i, (h, m, p, mk, lb) in enumerate(batch):
        t = h.shape[0]
        H[i, :t] = h.float()
        M[i, :t] = m.float()
        PR[i, :t] = p.float()
        MK[i, :t] = mk
        LB[i] = lb
    return H, M, PR, MK, LB

g = torch.Generator().manual_seed(eh.SEED)
BATCH_SIZE = max(1, min(BATCH_SIZE, max(1, len(tr) // 2)))   # tiny-data guard
train_loader = torch.utils.data.DataLoader(
    SeqDataset(tr), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate, generator=g, drop_last=True)
val_loader = torch.utils.data.DataLoader(
    SeqDataset(va or ho), batch_size=BATCH_SIZE, collate_fn=collate)
assert len(train_loader) > 0 and len(val_loader) > 0, \
    f"empty loader: {len(tr)} train / {len(va or ho)} val clips, batch {BATCH_SIZE}"
print(f"batches: train {len(train_loader)} | val {len(val_loader)} (batch {BATCH_SIZE})")


In [ ]:
model = eh.EmotionFusionHead()
print(f"head params: {model.count_params()/1e6:.2f}M")
best_f1, history = eh.train_head(model, train_loader, val_loader, device=DEVICE,
                                 epochs=EPOCHS, lr=LR, patience=PATIENCE)
eh.save_head(model, P.MODELS / "head_phaseA.pt",
             extra={"val_macro_f1": best_f1, "history": history,
                    "smoke_test": bool(P.SMOKE)})
print("saved:", P.MODELS / "head_phaseA.pt", "| best val macro-F1:", round(best_f1, 4))
gate2 = best_f1 >= max(f1_helium, f1_mimi, f1_pros) + GATE2_MIN_GAIN
print(f"GATE 2 (fusion >= best probe + {GATE2_MIN_GAIN}):", "PASS" if gate2 else "FAIL")
if GATE_STRICT:
    assert gate2


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

eval_recs2 = ho or va
hold_loader = torch.utils.data.DataLoader(SeqDataset(eval_recs2), batch_size=BATCH_SIZE,
                                          collate_fn=collate)
logits, labels, masks = eh.evaluate_head(model, hold_loader, DEVICE)
pred = logits.argmax(-1)
yt, yp = [], []
for i in range(logits.shape[0]):
    m = masks[i]
    yt += labels[i].expand(logits.shape[1])[m].tolist()
    yp += pred[i][m].tolist()
print(classification_report(yt, yp, labels=list(range(len(eh.EMOTIONS))),
                            target_names=eh.EMOTIONS, digits=3, zero_division=0))
cm = confusion_matrix(yt, yp, normalize="true")
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(4), eh.EMOTIONS)
ax.set_yticks(range(4), eh.EMOTIONS)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("Phase A holdout (agent frames)")
for i in range(4):
    for j in range(4):
        ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center")
P.REPORTS.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(P.REPORTS / "phaseA_confusion.png", dpi=130)
plt.show()


In [ ]:
# user-emotion leakage: predicted AGENT emotion must NOT track the USER's emotion
leak = []
for i, r in enumerate(eval_recs2):
    mk = r["agent_mask"]
    if int(r.get("user_emotion", -1)) >= 0 and mk.any():
        votes = pred[i][: len(mk)][mk].mode().values   # pred is padded to batch T
        leak.append((int(r["user_emotion"]), int(votes)))
if leak:
    uy = [a for a, _ in leak]
    up = [b for _, b in leak]
    from sklearn.metrics import f1_score as _f1
    print(f"leakage probe (user emotion vs predicted agent emotion; chance~0.25): "
          f"macro-F1={_f1(uy, up, average='macro'):.3f} on {len(leak)} clips")
else:
    print("no user_emotion labels in the selection — leakage test skipped")

# calibration: ECE before/after temperature scaling (T fit on the val split)
def ece(logits_t, labels_t, masks_t, n_bins=15):
    probs = torch.softmax(logits_t, -1)
    conf, pr = probs.max(-1)
    cs, ys, prs = [], [], []
    for i in range(logits_t.shape[0]):
        m = masks_t[i]
        if m.any():
            cs.append(conf[i][m])
            ys.append(labels_t[i].expand(logits_t.shape[1])[m])
            prs.append(pr[i][m])
    c = torch.cat(cs); y = torch.cat(ys); p = torch.cat(prs)
    e = 0.0
    for b in range(n_bins):
        lo, hi = b / n_bins, (b + 1) / n_bins
        m = (c > lo) & (c <= hi)
        if m.any():
            e += float(m.float().mean()) * abs(float((p[m] == y[m]).float().mean()) - float(c[m].mean()))
    return e

val_logits, val_labels, val_masks = eh.evaluate_head(model, val_loader, DEVICE)
T_log = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.LBFGS([T_log], lr=0.05, max_iter=100)
def closure():
    opt.zero_grad()
    scaled = val_logits / T_log.exp()
    tgt = val_labels.unsqueeze(1).expand(-1, val_logits.shape[1])
    loss = torch.nn.functional.cross_entropy(scaled[val_masks], tgt[val_masks])
    loss.backward()
    return loss
opt.step(closure)
temperature = float(T_log.exp())
print(f"ECE raw={ece(val_logits, val_labels, val_masks):.4f} -> "
      f"T-scaled={ece(val_logits / temperature, val_labels, val_masks):.4f} (T={temperature:.3f})")

# switching behaviour with the hysteresis smoother
sm = eh.HysteresisSmoother()
sw = []
for i in range(logits.shape[0]):
    m = masks[i]
    if not m.any():
        continue
    sm.reset()
    for t in range(logits.shape[1]):
        if m[t]:



            sm.update(torch.softmax(logits[i, t].float(), -1).numpy())
    sw.append(sm.switches_per_min())
if sw:
    print(f"smoothed switches/min: mean={np.mean(sw):.2f} p95={np.percentile(sw, 95):.2f} "
          f"(target < 0.5)")
else:
    print("no masked eval frames — switching report skipped")


In [ ]:
metrics = dict(
    gate1=dict(helium=f1_helium, mimi=f1_mimi, prosody=f1_pros, passed=bool(gate1)),
    gate2=dict(val_macro_f1=best_f1, passed=bool(gate2)),
    temperature=temperature, history=history)
(P.REPORTS / "phaseA_metrics.json").write_text(json.dumps(metrics, indent=2))
print("wrote", P.REPORTS / "phaseA_metrics.json")
print("→ continue with 04_generate_indomain_emotionplex7b.ipynb")
